<a href="https://colab.research.google.com/github/lannd3217/Interview_RAG/blob/main/RAG_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/lannd3217/Interview_RAG/

In [ ]:
%cd Interview_RAG

In [ ]:
%%capture

%%capture
!pip install -q ragas datasets
!pip install -U langchain langchain-community langchain-openai
!pip install -U langchain langchain-community
!pip install -U pymupdf langchain-community
!pip install -U langchain-huggingface sentence-transformers
!pip install langchain langchain-community langchain-chroma langchain-huggingface pymupdf sentence-transformers
# !pip install chromadb
!pip install -q "chromadb>=0.5.0"


In [ ]:
# !pip install -q "chromadb==0.4.24" "opentelemetry-api==1.24.0" "opentelemetry-sdk==1.24.0"


In [ ]:
## LOAD VECTOR STORE AND BUILD RETRIEVER

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PERSIST_DIR = "./interview_vector_db"
COLLECTION_NAME = "interview_prep_collection"

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)

vector_store = Chroma(
    persist_directory="./interview_vector_db",
    embedding_function=embeddings,
    collection_name = "interview_prep_collection"
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2} # Fetches top-2 most relevant chunks
)


In [ ]:

from transformers import AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import torch, warnings

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
)

llm = HuggingFacePipeline(pipeline=pipe)

def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def make_chat_prompt(context: str, question: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an interview preparation assistant. "
                "Answer only using the provided context. "
                "If the answer is not in the context, say \"I don't know.\""
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def to_chat_prompt(inputs: dict) -> str:
    return make_chat_prompt(inputs["context"], inputs["question"])

def clean_answer(text: str) -> str:
    # Keep only after the assistant tag if present
    if "<|assistant|>" in text:
        text = text.split("<|assistant|>", 1)[1].strip()
    # Strip trailing special tokens
    text = text.split("</s>")[0].strip()
    # Keep first 2–3 sentences
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return ". ".join(sentences) + ("." if sentences else "")


chain = (
    {
        "context": retriever | RunnableLambda(combine_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(to_chat_prompt)   # build TinyLlama chat prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(clean_answer)
)

# Quick test
print(chain.invoke("What is an effective strategy for choosing what to study before a technical interview?"))

In [ ]:
import json

with open("ragas_eval_dataset.json") as f:
    eval_data= json.load(f)


# question = "What is an effective strategy for choosing what to study before a technical interview?"
item = eval_data[0]
print("Q:", item["question"])
print("A:", item["answer"])


In [ ]:
from ipywidgets import widgets
from IPython.display import display, clear_output


questions = [item["question"] for item in eval_data] + ["Custom question..."]

dropdown = widgets.Dropdown(options=questions, description="Question:", layout=widgets.Layout(width="700px"))
custom_input = widgets.Text(placeholder="Type your own question here...", layout=widgets.Layout(width="700px"))
button = widgets.Button(description="Ask", button_style="primary")
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        q = custom_input.value.strip() if dropdown.value == "Custom question..." else dropdown.value
        print(f"Q: {q}\n")
        print("Thinking...")
        # clear_output(wait=True)
        answer = chain.invoke(q)
        # print(f"Q: {q}\n")
        print(f"A: {answer}")

button.on_click(on_click)
display(dropdown, custom_input, button, output)


In [ ]:
# demo questions
# How do I stay competitive given I have no prior data science experience
# How do I stand out in a behavioral interview
# Do I need a masters to get into Data Science roles
